# Basic Label Flipping Attack on a Binary Classifier

**Author:** Teodora Demerdzhieva  
**Topic:** Adversarial Machine Learning - Data Poisoning  
**Difficulty:** Beginner

---

## What This Notebook Covers

This notebook demonstrates a **label flipping attack** against a binary logistic regression classifier.

Label flipping is the simplest form of data poisoning:
- No features are modified
- Only the **labels** (the correct answers) are changed
- The model trains on wrong information and learns incorrect decision boundaries
- The result: a model that performs far worse than it should

It's the equivalent of telling someone the wrong answers while they're studying for an exam - the information looks legitimate, but it's deliberately wrong.

---

## How It Differs from the Targeted Version

| | Basic Label Flipping | Targeted Label Flipping |
|-|-|-|
| Which labels are flipped? | Random across all samples | Only one specific class |
| Goal | Degrade overall accuracy | Degrade one class while keeping others intact |
| Detection difficulty | Easier - overall accuracy drops | Harder - looks like a data quality issue |
| Classes | Binary (2 classes) | Multi-class |

---

## Attack Goal

> Flip 60% of the training labels randomly so the model learns a completely wrong decision boundary, reducing its accuracy from ~99% to near random chance.

---

## How It Works - The Big Picture

Imagine a spam filter trained on emails labeled "spam" or "not spam".

A label flipping attack randomly swaps 60% of those labels before training:
- Some spam emails get labeled as "not spam"
- Some legitimate emails get labeled as "spam"

The model trains on this corrupted data and learns a boundary that is almost the opposite of correct. At test time, it classifies most things wrong - not because it's a bad model, but because it was taught wrong.

## 1. Setup and Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import json
import requests
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Fixed parameters - must match what the evaluator expects
POISON_RATE = 0.60   # flip 60% of all training labels
SEED        = 1337   # random seed for reproducibility

print(f"Poison rate : {POISON_RATE*100:.0f}% of training labels will be flipped")
print(f"Random seed : {SEED}")

## 2. Load the Dataset

The dataset is a binary 2D classification problem (two classes, two features, already standardized).

> **Note:** The dataset file (`label_flipping_dataset.npz`) is not included in this repository as it is proprietary to the course. To reproduce this notebook, provide your own binary 2D dataset in `.npz` format with keys: `Xtr`, `ytr`, `Xte`, `yte`.

In [ ]:
dataset_filename = "label_flipping_dataset.npz"

data    = np.load(dataset_filename)
X_train = data["Xtr"]
y_train = data["ytr"]
X_test  = data["Xte"]
y_test  = data["yte"]
data.close()

print(f"Training set : {X_train.shape}")
print(f"Test set     : {X_test.shape}")
print(f"Classes      : {np.unique(y_train)}")
print(f"\nClass distribution in training set:")
for cls in np.unique(y_train):
    count = np.sum(y_train == cls)
    print(f"  Class {cls}: {count} samples ({count/len(y_train)*100:.1f}%)")

## 3. Visualize the Clean Training Data

Before poisoning, we visualize the dataset to understand the natural class separation.

The two classes are well separated.

In [ ]:
def plot_dataset(X, y, title, model=None):
    """
    Scatter plot of a 2D binary dataset.
    Optionally overlays the decision boundary of a trained model.
    """
    colors = ["#0086ff", "#ffaf00"]   # blue for class 0, yellow for class 1

    plt.figure(figsize=(10, 6))
    for cls in np.unique(y):
        mask = y == cls
        plt.scatter(X[mask, 0], X[mask, 1],
                    c=colors[int(cls)], label=f"Class {int(cls)}",
                    edgecolors="#141d2b", s=50, alpha=0.7)

    # Draw decision boundary if a model is provided
    if model is not None:
        x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
        y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
        xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                             np.linspace(y_min, y_max, 200))
        Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
        plt.contourf(xx, yy, Z, alpha=0.15,
                     cmap=plt.cm.colors.ListedColormap(colors))
        plt.contour(xx, yy, Z, colors="white", linewidths=1, linestyles="--")

    plt.title(title, fontsize=14)
    plt.xlabel("Feature 1 (standardized)")
    plt.ylabel("Feature 2 (standardized)")
    plt.legend()
    plt.tight_layout()
    plt.show()


plot_dataset(X_train, y_train, "Clean Training Data")

## 4. Train a Baseline Model (Before Attack)

Before the attack, a baseline model confirms the data is cleanly separable.

In [ ]:
baseline_model = LogisticRegression(random_state=SEED)
baseline_model.fit(X_train, y_train)

y_pred_baseline = baseline_model.predict(X_test)
baseline_acc    = accuracy_score(y_test, y_pred_baseline)

print("=== Baseline Model (Clean Data) ===")
print(f"Test accuracy: {baseline_acc:.4f}")
print()
print(classification_report(
    y_test, y_pred_baseline,
    target_names=["Class 0", "Class 1"]
))

plot_dataset(X_train, y_train,
             "Baseline Model - Decision Boundary on Clean Data",
             model=baseline_model)

## 5. The Label Flipping Attack

### Strategy

We randomly select 60% of all training samples and flip their labels:

```
Label 0 → becomes Label 1
Label 1 → becomes Label 0
```

This is done using the formula `1 - label` - simple for binary labels.

### Why 60%?

Flipping more than 50% crosses a threshold where the model learns a boundary that is close to the **inverse** of the correct one. At 60%:

```
40% of training data → correct labels
60% of training data → wrong labels

The wrong labels dominate → model learns the wrong boundary
→ accuracy drops to near random chance
```

Below 50%, the correct labels still dominate and the model mostly learns correctly - just with reduced confidence.

In [ ]:
def flip_labels(y, poison_rate, seed):
    """
    Randomly flips a fraction of binary labels in the training set.

    For binary labels, flipping means: 0 → 1 and 1 → 0.
    The fraction flipped is controlled by poison_rate.
    A fixed seed ensures reproducibility.

    Parameters
    ----------
    y           : original label array (binary: 0 or 1)
    poison_rate : fraction of labels to flip (0.0 to 1.0)
    seed        : random seed for reproducibility

    Returns
    -------
    y_poisoned   : label array with flipped labels
    flipped_idx  : indices of the flipped samples
    """
    np.random.seed(seed)

    # Work on a copy - never modify the original array
    y_poisoned = y.copy()

    # Calculate how many labels to flip
    n_samples = len(y)
    n_flip    = int(poison_rate * n_samples)

    # Randomly choose which indices to flip
    flipped_idx = np.random.choice(n_samples, n_flip, replace=False)

    # Flip binary labels: 0 becomes 1, 1 becomes 0
    y_poisoned[flipped_idx] = 1 - y_poisoned[flipped_idx]

    return y_poisoned, flipped_idx


# Run the attack
y_train_poisoned, flipped_idx = flip_labels(y_train, POISON_RATE, SEED)

print(f"Total training samples  : {len(y_train)}")
print(f"Labels flipped          : {len(flipped_idx)} ({POISON_RATE*100:.0f}%)")
print()
print(f"Label distribution before: {np.bincount(y_train)}")
print(f"Label distribution after : {np.bincount(y_train_poisoned)}")

## 6. Visualize the Poisoned Data

The scatter plot below shows the result. Labels no longer reflect the actual data structure. The model sees contradictory 

examples everywhere and cannot learn a meaningful boundary.

In [ ]:
plot_dataset(
    X_train, y_train_poisoned,
    title=f"Poisoned Training Data - {len(flipped_idx)} labels flipped ({POISON_RATE*100:.0f}%)"
)

## 7. Train the Poisoned Model and Evaluate

Training on poisoned data uses the same architecture - the model has no way to distinguish a flipped label from a real one.

In [ ]:
poisoned_model = LogisticRegression(random_state=SEED)
poisoned_model.fit(X_train, y_train_poisoned)

y_pred_poisoned = poisoned_model.predict(X_test)
poisoned_acc    = accuracy_score(y_test, y_pred_poisoned)

print("=== Poisoned Model Results ===")
print(f"Baseline accuracy : {baseline_acc:.4f}")
print(f"Poisoned accuracy : {poisoned_acc:.4f}")
print(f"Accuracy drop     : {baseline_acc - poisoned_acc:.4f}")
print()
print(classification_report(
    y_test, y_pred_poisoned,
    target_names=["Class 0", "Class 1"],
    zero_division=0
))

plot_dataset(
    X_train, y_train_poisoned,
    title=f"Poisoned Model - Decision Boundary (accuracy: {poisoned_acc:.4f})",
    model=poisoned_model
)

if poisoned_acc < 0.10:
    print("Attack SUCCESSFUL - model accuracy near zero")
elif poisoned_acc < 0.50:
    print("Attack SUCCESSFUL - accuracy significantly degraded")
else:
    print("Attack insufficient - try increasing POISON_RATE")

## 8. Key Takeaways

| Aspect | Detail |
|--------|--------|
| **Attack type** | Random label flipping |
| **Flip rate** | 60% of all training labels |
| **Features modified** | None - only labels changed |
| **Model** | Binary Logistic Regression |
| **Effect** | Accuracy drops from ~99% to near 0% |
| **Detection difficulty** | Medium - overall accuracy drop is visible, but cause is not obvious |

---

### Why Flipping More Than 50% Is So Effective

At exactly 50% flipped, the model sees equal amounts of correct and incorrect labels - it essentially learns nothing and predicts randomly.

At 60% flipped, the incorrect labels **dominate**. The model confidently learns the wrong boundary - it becomes a model that is reliably wrong, which is arguably worse than random.

---

### Real-World Implications

Label flipping attacks are a real concern in any system where training labels come from outside sources:
- Crowdsourced annotation platforms (a malicious annotator)
- Active learning pipelines (manipulating the labels fed back into training)
- Federated learning (a malicious participant sends wrong labels)
- Web-scraped datasets where labels are inferred from metadata

### Defenses

- **Label consistency checks** - flag samples where the label contradicts the feature values
- **Confident learning** - automatically detect likely mislabeled samples before training
- **Data provenance** - track who labeled each sample and audit suspicious annotators
- **Robust loss functions** - training objectives that are less sensitive to noisy labels
- **Ensemble disagreement** - if multiple models disagree on a sample, investigate the label

---

### References

- [MITRE ATLAS - Poison Training Data](https://atlas.mitre.org/techniques/AML.T0020)
- [OWASP Machine Learning Security Top 10](https://owasp.org/www-project-machine-learning-security-top-10/)